# Geometry 05 — Prouvé par Gröbner, prouvé par le noyau

**Public : Recherche** — cinquième et dernière position du chemin principal de la série [Geometry](README.md), le programme gradué de la preuve automatique en géométrie. Il suppose la lecture de 01 (*De la figure à l'équation*), 02 (*Prouver par l'algèbre*) et 03 (*La méthode de Wu*), ainsi qu'une première rencontre avec la série [Lean](../README.md) — le carnet [Lean-38](../../Lean-38-Capstone-Geometry-Lean-Python.ipynb) en est l'escalier d'entrée.

**Ce que ce notebook suppose.** L'encodage polynomial des hypothèses et l'appartenance à un idéal (01, 02), l'idée qu'une méthode de preuve produit un *objet vérifiable* (02, 03), et la lecture d'un fichier source en Python. Aucune connaissance de Mathlib ni de Lean n'est requise : le carnet lit les énoncés du lac, il ne reconstruit pas ses preuves et ne lance aucun build.

***

Les positions 02 et 03 ont prouvé le fil rouge de la série — le milieu de l'hypoténuse équidistant des trois sommets — par un **calcul** : l'appartenance du polynôme de conclusion à l'idéal des hypothèses donne un **reste nul**. La position 05 pose la question que le programme gradué a gardée pour la fin :

> Que garantit exactement « prouvé par Gröbner » — et qu'est-ce qu'une preuve du **noyau** garantit de plus ?

Les deux machines ont traité le même théorème. Elles ne délivrent pas le même objet, et l'écart se mesure. Le lac `geometry_lean` (EPIC #18601, volet B) porte la version formelle : quatre énoncés Lean qui prouvent le fil rouge dans un espace à produit scalaire quelconque, sans coordonnées.

Ce carnet tient les deux preuves côte à côte, sur le **même** théorème, et mesure trois choses : ce que le reste nul certifie (§2, §3), où un énoncé faux est rejeté (§4), et ce qui arrive au verdict algébrique quand l'encodage change (§5).

## §1. Le fil rouge, tel que le lac l'énonce

Le lac `geometry_lean` vit sous `Geometry/geometry_lean/`. Son module central est `Geometry/MidpointHypotenuse.lean`. On le lit **comme un fichier** : c'est le pattern établi par l'escalier d'entrée de la série, et il évite de dépendre d'un build Lean local pour lire des énoncés.

In [1]:
import os
import re

# Le carnet vit dans Geometry/ ; le lake est son sous-dossier. Les candidats
# couvrent l'execution depuis la racine du depot comme depuis le dossier serie.
CANDIDATS = [
    "geometry_lean/Geometry/MidpointHypotenuse.lean",
    "Geometry/geometry_lean/Geometry/MidpointHypotenuse.lean",
    os.path.join("MyIA.AI.Notebooks", "SymbolicAI", "Lean", "Geometry",
                 "geometry_lean", "Geometry", "MidpointHypotenuse.lean"),
]


def chemin_lac():
    for candidat in CANDIDATS:
        if os.path.exists(candidat):
            return candidat
    return None


chemin = chemin_lac()
print("fichier du lac :", chemin)
if chemin is None:
    raise FileNotFoundError(
        "MidpointHypotenuse.lean introuvable : executer depuis Geometry/ ou la racine du depot"
    )

texte_lac = open(chemin, encoding="utf-8").read()

# Les enonces : un theoreme par bloc `theorem nom ... :=` (ou `private theorem`).
ENONCE = re.compile(r"^(private\s+)?theorem\s+(\w+)(.*?):=", re.S | re.M)
enonces = ENONCE.findall(texte_lac)
print("enonces trouves :", len(enonces))
for prive, nom, signature in enonces:
    tete = " ".join(signature.split())
    print(f"  [{'prive' if prive else 'public'}] {nom}")
    print(f"      {tete[:150]}")

fichier du lac : geometry_lean/Geometry/MidpointHypotenuse.lean
enonces trouves : 4
  [public] norm_add_eq_norm_sub_of_inner_eq_zero
      {u v : E} (huv : ⟪u, v⟫_ℝ = 0) : ‖u + v‖ = ‖u - v‖
  [prive] dist_origin_eq_dist_apex_of_inner_eq_zero
      {u v : E} (huv : ⟪u, v⟫_ℝ = 0) : ‖(2 : ℝ)⁻¹ • (u + v)‖ = ‖u - (2 : ℝ)⁻¹ • (u + v)‖
  [public] midpoint_hypotenuse_equidistant
      {u v : E} (huv : ⟪u, v⟫_ℝ = 0) : ‖(2 : ℝ)⁻¹ • (u + v)‖ = ‖u - (2 : ℝ)⁻¹ • (u + v)‖ ∧ ‖(2 : ℝ)⁻¹ • (u + v)‖ = ‖v - (2 : ℝ)⁻¹ • (u + v)‖
  [public] midpoint_hypotenuse_radius
      {u v : E} (huv : ⟪u, v⟫_ℝ = 0) : ‖(2 : ℝ)⁻¹ • (u + v)‖ = (2 : ℝ)⁻¹ * ‖u - v‖


### Lecture du résultat

Quatre énoncés, dont un privé. Le lac ne prouve pas « le théorème du milieu de l'hypoténuse » d'un bloc : il le décompose en une chaîne.

| Énoncé | Rôle dans la chaîne |
|---|---|
| `norm_add_eq_norm_sub_of_inner_eq_zero` | les diagonales d'un parallélogramme **rectangle** ont même norme — c'est Pythagore sous forme additive et soustractive |
| `dist_origin_eq_dist_apex_of_inner_eq_zero` (privé) | la composante « sommet B » : elle ne sert qu'à l'énoncé suivant |
| `midpoint_hypotenuse_equidistant` | **le fil rouge** : `MA = MB ∧ MA = MC` |
| `midpoint_hypotenuse_radius` | le rayon vaut la moitié de l'hypoténuse |

Le point à retenir pour la suite : ces énoncés ne mentionnent **aucune coordonnée**. `variable {E : Type*} [NormedAddCommGroup E] [InnerProductSpace ℝ E]` — le théorème vaut dans tout espace à produit scalaire réel. C'est ce que §5 confrontera à l'encodage algébrique.

## §2. Le même fil rouge, prouvé par un calcul

On reprend l'encodage des positions 02 et 03, pour que les verdicts soient comparables : le sommet de l'angle droit est à l'origine, `B = (0, 0)`, les deux autres sommets sont `A = (xa, ya)` et `C = (xc, yc)`, et le milieu `M = (xm, ym)` est une **variable libre** — les équations de milieu font partie des hypothèses au lieu d'être substituées.

Les trois hypothèses et les trois conclusions du fil rouge :

| Objet | Polynôme |
|---|---|
| hypothèse `BA ⊥ BC` | `xa*xc + ya*yc` |
| hypothèse `M milieu de [AC]` | `2*xm - xa - xc` et `2*ym - ya - yc` |
| conclusion `MA² = MB²` | `(xm-xa)² + (ym-ya)² - (xm² + ym²)` |
| conclusion `MA² = MC²` | `(xm-xa)² + (ym-ya)² - ((xm-xc)² + (ym-yc)²)` |
| conclusion rayon `MB² = AC²/4` | `xm² + ym² - ((xa-xc)² + (ya-yc)²)/4` |

In [2]:
import sympy as sp

xa, ya, xc, yc, xm, ym = sp.symbols("xa ya xc yc xm ym", real=True)
VARS = (xa, ya, xc, yc, xm, ym)

perp = xa * xc + ya * yc
mid1 = 2 * xm - xa - xc
mid2 = 2 * ym - ya - yc
HYPOTHESES = [perp, mid1, mid2]

MA2 = (xm - xa) ** 2 + (ym - ya) ** 2
MB2 = xm**2 + ym**2
MC2 = (xm - xc) ** 2 + (ym - yc) ** 2
AC2 = (xa - xc) ** 2 + (ya - yc) ** 2

# Le domaine doit etre QQ : les conclusions portent des coefficients 1/4.
base = sp.groebner(HYPOTHESES, *VARS, order="lex", domain=sp.QQ)
print("base de Groebner :", len(base.polys), "polynomes")


def reste(polynome):
    """Reste de la division du polynome par la base de Groebner des hypotheses."""
    return sp.simplify(base.reduce(sp.expand(polynome))[1])


for nom, conclusion in [("MA^2 = MB^2", MA2 - MB2),
                        ("MA^2 = MC^2", MA2 - MC2),
                        ("rayon : MB^2 = AC^2/4", MB2 - AC2 / 4)]:
    print(f"  reste de ({nom}) : {reste(conclusion)}")

base de Groebner : 3 polynomes
  reste de (MA^2 = MB^2) : 0
  reste de (MA^2 = MC^2) : 0
  reste de (rayon : MB^2 = AC^2/4) : 0


### Lecture du résultat

Trois conclusions distinctes, **trois restes nuls**. C'est le verdict des positions 02 et 03, reproduit ici : le fil rouge est vrai, et la méthode le certifie.

Le contrôle qui compte est ailleurs. Le même encodage appliqué à un triangle **non rectangle** doit rendre un reste non nul — sinon la méthode ne prouverait rien. C'est l'objet du §4.

Une remarque de méthode : `domain=sp.QQ` est nécessaire. Sans lui, sympy construit la base sur les entiers et refuse les coefficients `1/4` de la conclusion sur le rayon — l'erreur ne dit pas « le théorème est faux », elle dit « le corps de coefficients ne les contient pas ». Un piège à connaître quand on juge un verdict algébrique.

## §3. Le certificat : ce que « reste nul » veut dire

Un reste nul n'est pas un verdict, c'est l'annonce d'un **objet** : les cofacteurs. La division d'un polynôme par une base de Gröbner rend, avec le reste, des cofacteurs `c_1, ..., c_k` tels que

`conclusion = c_1 * g_1 + ... + c_k * g_k + reste`

avec les `g_i` les générateurs de la base. Quand le reste est nul, cette somme **est** un certificat : n'importe qui peut remultiplier et vérifier, sans refaire la division. C'est l'analogue exact du terme de preuve que le noyau vérifiera au §6.

In [3]:
cofacteurs, reste_final = base.reduce(sp.expand(MA2 - MB2))

print("cofacteurs du certificat :")
for c, g in zip(cofacteurs, base.polys):
    if sp.simplify(c) != 0:
        print(f"  ({sp.simplify(c)}) * ({g.as_expr()})")

reconstitution = sp.expand(sum(c * g.as_expr() for c, g in zip(cofacteurs, base.polys)))
ecart = sp.simplify(reconstitution - (MA2 - MB2))
print("reste de la division :", sp.simplify(reste_final))
print("reconstitution exacte ?", ecart == 0)
print("ecart :", ecart)

cofacteurs du certificat :
  (xa - xc) * (xa + xc - 2*xm)
  (ya - yc) * (ya + yc - 2*ym)
  (1) * (xc**2 - 2*xc*xm + yc**2 - 2*yc*ym)
reste de la division : 0
reconstitution exacte ? True
ecart : 0


### Lecture du résultat

Le certificat tient en trois cofacteurs — `(xa - xc)`, `(ya - yc)` et `1` — et la remultiplication redonne la conclusion **exactement** (écart nul). Le fait est vérifiable par un lecteur qui ne fait confiance ni à sympy ni à ce carnet.

Deux choses à voir dans cette liste de cofacteurs :

- **la preuve n'est pas l'hypothèse.** Un des cofacteurs vaut `1` : la conclusion se sert directement d'un générateur. Les deux autres portent les équations de milieu, avec des coefficients `(xa - xc)` et `(ya - yc)` — ce sont eux qui encodent *où* est le milieu. Retirer les équations de milieu du jeu d'hypothèses casse la preuve, et le §5 montre que le verdict bascule alors.
- **le certificat dépend du jeu d'hypothèses choisi.** Ce n'est pas une faiblesse de sympy : c'est la nature de la méthode. Un reste nul certifie une appartenance à **un** idéal, et l'idéal est une décision de celui qui encode.

## §4. Témoin négatif : l'énoncé faux, et où il est rejeté

Une méthode de preuve qui dit « oui » à tout ne prouve rien. Le témoin négatif du programme gradué : l'énoncé **faux** `MA² = MB² / 4`, sur le même encodage et les mêmes hypothèses.

In [4]:
FAUX = MA2 - MB2 / 4
reste_faux = reste(FAUX)
print("reste de l'enonce faux (MA^2 = MB^2/4) :", reste_faux)

# Le reste est un POLYNOME : il s'annule sur sa variete. Laquelle, exactement ?
# Sa forme factorisee le dit sans ambiguite.
print("forme factorisee :", sp.factor(reste_faux))
# C'est (3/4)*(xm^2 + ym^2) -- une somme de deux carres. Sur les REELS, elle ne
# s'annule donc qu'en xm = ym = 0, c'est-a-dire M confondu avec B.
somme_de_carres = sp.simplify(
    sp.factor(reste_faux) - sp.Rational(3, 4) * (xm**2 + ym**2)
) == 0
print("le reste vaut (3/4)*(xm^2 + ym^2) ?", somme_de_carres)
print("valeur en (xm, ym) = (0, 0) :", sp.expand(reste_faux).subs({xm: 0, ym: 0}))
# sympy resout par defaut dans C et parametre donc par des ym imaginaires :
# "ou le reste s'annule-t-il" n'a pas de reponse sans dire sur quel corps on
# pose la question. C'est le point du paragraphe, et il se lit dans la sortie.
print("racines rendues par sympy (corps par defaut : C) :",
      sp.solve(sp.Eq(reste_faux, 0), xm, ym))

# Contre-exemple numerique : le triangle 3-4-5, B=(0,0), A=(3,0), C=(0,4), M=(1.5,2)
triangle = {xa: 3, ya: 0, xc: 0, yc: 4, xm: sp.Rational(3, 2), ym: 2}
valeurs = {nom: sp.expand(expr).subs(triangle) for nom, expr in
           [("MA^2", MA2), ("MB^2", MB2), ("MC^2", MC2), ("MB^2/4", MB2 / 4)]}
for nom, valeur in valeurs.items():
    print(f"  {nom:8s} = {valeur}")
print("ecart de l'enonce faux sur ce triangle :",
      sp.expand(FAUX).subs(triangle))


reste de l'enonce faux (MA^2 = MB^2/4) : 3*xm**2/4 + 3*ym**2/4
forme factorisee : 3*(xm**2 + ym**2)/4
le reste vaut (3/4)*(xm^2 + ym^2) ? True
valeur en (xm, ym) = (0, 0) : 0
racines rendues par sympy (corps par defaut : C) : [(-I*ym, ym), (I*ym, ym)]
  MA^2     = 25/4
  MB^2     = 25/4
  MC^2     = 25/4
  MB^2/4   = 25/16
ecart de l'enonce faux sur ce triangle : 75/16


### Lecture du résultat

Le reste est **non nul** : `3*xm²/4 + 3*ym²/4`. L'énoncé faux est rejeté.

La lecture fine est plus intéressante que le rejet. Ce reste s'annule quand `xm = ym = 0`, c'est-à-dire quand `M` est confondu avec `B` — la configuration **dégénérée** où `A`, `B` et `C` sont alignés. Autrement dit : la méthode rejette l'énoncé faux sur toute la variété **générique**, et ne le rejette pas sur une sous-variété dégénérée où l'énoncé est vrai par accident.

Un mot sur le corps, parce que la sortie le met sous les yeux : `sympy` résout par défaut dans `ℂ` et rend `xm = ±i·ym` — des racines qui n'existent pas dans le plan euclidien. Sur les **réels**, `(3/4)(xm² + ym²)` est une somme de deux carrés, donc nulle à la seule origine. « Où le reste s'annule-t-il ? » n'a pas de réponse sans dire **sur quel corps** on pose la question : c'est le même avertissement que « reste non nul », un cran plus loin.

C'est exactement le point que la position 05 existe pour montrer, et il se lit dans le verdict lui-même : le résidu d'un calcul algébrique est un **polynôme**, pas un booléen. « Reste non nul » veut dire « non nul sauf sur la variété où ce polynôme s'annule ». La question « et si cette variété dégénérée contient la configuration qui m'intéresse ? » est celle que la méthode ne pose pas — et c'est précisément le rôle des conditions de non-dégénérescence dans 02 et 03.

Sur le triangle 3-4-5, le contre-exemple est net : `MA² = 6.25`, `MB²/4 = 1.5625`.


## §5. L'écart mesuré : l'encodage se porte, le noyau ne le porte pas

Voici le cœur de la position 05.

La preuve du §2 vit dans **un système de coordonnées**. Le changement de coordonnées le plus innocent — une rotation — devrait laisser le verdict intact, et c'est le cas. Mais la géométrie euclidienne n'est pas préservée par tous les changements de base, et c'est là que le verdict algébrique montre sa dépendance.

In [5]:
# 1. Rotation : la forme de l'hypothese est-elle invariante en coordonnees tournees ?
t = sp.symbols("t", real=True)
rotation = sp.Matrix([[sp.cos(t), -sp.sin(t)], [sp.sin(t), sp.cos(t)]])
A_tourne = rotation * sp.Matrix([xa, ya])
C_tourne = rotation * sp.Matrix([xc, yc])
perp_tourne = sp.expand((A_tourne.T * C_tourne)[0])
print("rotation  : hypothese inchangee ?", sp.simplify(perp_tourne - perp) == 0)

# 2. Cisaillement : (x, y) -> (x + a*y, y).  Ce n'est PAS une isometrie.
a = sp.symbols("a", real=True)
cisaille = sp.Matrix([[1, a], [0, 1]])
A_c = cisaille * sp.Matrix([xa, ya])
C_c = cisaille * sp.Matrix([xc, yc])
M_c = cisaille * sp.Matrix([xm, ym])
perp_c = sp.expand((A_c.T * C_c)[0])
print("cisaille  : hypothese inchangee ?", sp.simplify(perp_c - perp) == 0)
print("            forme cisaillee :", perp_c)

rotation  : hypothese inchangee ? True
cisaille  : hypothese inchangee ? False
            forme cisaillee : a**2*ya*yc + a*xa*yc + a*xc*ya + xa*xc + ya*yc


### Lecture du résultat

La rotation laisse `xa*xc + ya*yc` **inchangé** par identité. Le cisaillement, lui, remplace l'hypothèse par

`a²*ya*yc + a*xa*yc + a*xc*ya + xa*xc + ya*yc`

— ce n'est plus le même polynôme. Et c'est le point : l'orthogonalité **est** `xa*xc + ya*yc = 0` seulement dans une base orthonormée. Si l'encodeur oublie de transporter sa forme, l'idéal ne décrit plus les configurations qu'il croit décrire — sans que rien ne le signale.

Le code suivant mesure ce que ce décalage produit sur le verdict.

In [6]:
# 3. On rejoue le fil rouge sur les coordonnees cisailles, en gardant l'hypothese
#    ANCIENNE (u1*v1 + u2*v2 = 0).  L'encodeur a oublie de transporter la forme.
base_stale = sp.groebner(
    [perp,
     sp.expand(2 * M_c[0] - A_c[0] - C_c[0]),
     sp.expand(2 * M_c[1] - A_c[1] - C_c[1])],
    *VARS, order="lex")

MA2_c = sp.expand((M_c.T * M_c)[0])
MB2_c = sp.expand(((M_c - A_c).T * (M_c - A_c))[0])
reste_stale = sp.simplify(base_stale.reduce(sp.expand(MA2_c - MB2_c))[1])
print("reste avec l'hypothese NON transportee :", reste_stale)

# 4. Illustration numerique, cisaillement a = 1, sur le triangle 3-4-5.
sh = {xa: 3, ya: 0, xc: 0, yc: 4, xm: sp.Rational(3, 2), ym: 2}
A_pt = (sp.expand(A_c[0]).subs(sh), sp.expand(A_c[1]).subs(sh))
C_pt = (sp.expand(C_c[0]).subs(sh), sp.expand(C_c[1]).subs(sh))
M_pt = (sp.expand(M_c[0]).subs(sh), sp.expand(M_c[1]).subs(sh))
print("A, C, M cisailles (a=1) :", A_pt, C_pt, M_pt)
print("ancienne hypothese xA*xC + yA*yC :", sp.simplify(perp.subs(sh)))
print("|M|^2 :", sp.simplify((M_pt[0] ** 2 + M_pt[1] ** 2)))
print("|M-A|^2 :", sp.simplify(((M_pt[0] - A_pt[0]) ** 2 + (M_pt[1] - A_pt[1]) ** 2)))

reste avec l'hypothese NON transportee : a*(-a*yc**2 + 2*a*yc*ym - 2*xc*yc + 2*xc*ym + 2*xm*yc)
A, C, M cisailles (a=1) : (3, 0) (4*a, 4) (2*a + 3/2, 2)
ancienne hypothese xA*xC + yA*yC : 0
|M|^2 : (4*a + 3)**2/4 + 4
|M-A|^2 : (4*a - 3)**2/4 + 4


### Lecture du résultat

Le reste n'est **pas** nul : `a*(-a*yc² + 2*a*yc*ym - 2*xc*yc + 2*xc*ym + 2*xm*yc)`. La méthode algébrique déclare donc le fil rouge **faux** — alors que le théorème n'a pas changé d'un iota.

Ce que cela établit, en une phrase : **le verdict algébrique vaut relativement à un encodage, et il n'a aucun moyen de distinguer « le théorème est faux » de « votre forme d'hypothèse est périmée ».** Les deux se présentent identiquement — un reste non nul. L'illustration numérique le confirme : après cisaillement à `a = 1`, le point `M` n'est plus équidistant de `A` et de `B` (`|M|² ≠ |M-A|²`), non parce que le théorème a échoué, mais parce que la configuration cisaillée **n'est plus un triangle rectangle** — l'ancienne forme de l'hypothèse ne décrit plus l'orthogonalité.

Le noyau, lui, n'a pas ce problème — et pas parce qu'il est plus fort : **parce qu'il n'a pas d'encodage.** `⟪u, v⟫_ℝ = 0` ne nomme aucune coordonnée. Il n'y a pas de forme à transporter, donc pas de forme à périmer. La stabilité du noyau sous changement de base n'est pas une propriété qu'il possède *en plus* : c'est une conséquence de ce qu'il ne possède pas.

Le prix se paie ailleurs, et le programme gradué le dit dès la position 02 : le noyau ne vous dira pas **où** est le milieu, ni comment calculer le rayon. La preuve formelle est coordinate-free et, pour cette raison même, aveugle aux constructions. Une méthode algébrique calcule ; une preuve de noyau établit.

## §6. Ce que chaque machine garantit

Le tableau que la position 05 devait produire. Chaque ligne est mesurée dans ce carnet ou lisible dans le lac, pas reprise d'un manuel.

In [7]:
# Audit des axiomes interdits : compte TEXTUEL, indicateur seulement.
# La mesure qui fait autorite est l'organe du depot (scripts/lean/count_code_sorry.py,
# champ distinct_code_sorry) et la jambe CI `proof-integrity` du workflow lean-geometry.yml.
INTERDITS = ("sorry", "native_decide", "admit")
comptes = {mot: len(re.findall(r"\b" + mot + r"\b", texte_lac)) for mot in INTERDITS}
print("compte textuel dans le module du lac :", comptes)

print()
print("=== les deux verdicts, sur le meme theoreme ===")
print(f"{'':34s} | {'Grobner (02/03)':26s} | {'noyau Lean (05)'}")
print("-" * 96)
lignes = [
    ("objet produit", "un certificat (cofacteurs)", "un terme de preuve"),
    ("ce qui le verifie", "remultiplication par un lecteur", "le noyau, automatiquement"),
    ("hypotheses", "les generateurs de l'ideal", "une hypothese, quantifiee"),
    ("conditions de degenerescence", "a ajouter par l'encodeur", "aucune ici"),
    ("dependance a l'encodage", "oui -- §5 la mesure", "aucune -- sans coordonnees"),
    ("contre-exemple", "reste non nul -- §4", "enonce inprovable"),
    ("ce qu'il ne dit pas", "qu'il a bien encode", "ou est le milieu"),
]
for gauche, g, l in lignes:
    print(f"{gauche:34s} | {g:26s} | {l}")
print()
print("reste du fil rouge par Grobner :", reste(MA2 - MB2))
print("theoreme du lac                :", enonces[2][1] if len(enonces) > 2 else "?")

compte textuel dans le module du lac : {'sorry': 0, 'native_decide': 0, 'admit': 0}

=== les deux verdicts, sur le meme theoreme ===
                                   | Grobner (02/03)            | noyau Lean (05)
------------------------------------------------------------------------------------------------
objet produit                      | un certificat (cofacteurs) | un terme de preuve
ce qui le verifie                  | remultiplication par un lecteur | le noyau, automatiquement
hypotheses                         | les generateurs de l'ideal | une hypothese, quantifiee
conditions de degenerescence       | a ajouter par l'encodeur   | aucune ici
dependance a l'encodage            | oui -- §5 la mesure        | aucune -- sans coordonnees
contre-exemple                     | reste non nul -- §4        | enonce inprovable
ce qu'il ne dit pas                | qu'il a bien encode        | ou est le milieu

reste du fil rouge par Grobner : 0
theoreme du lac                : midpoint

### Lecture du résultat

Le compte textuel des mots-clés interdits est un **indicateur**, pas une mesure : il compte aussi la prose et les commentaires. La mesure qui fait autorité est celle de l'organe du dépôt (`scripts/lean/count_code_sorry.py`, champ `distinct_code_sorry`, qui ignore la prose) et de la jambe `proof-integrity` du workflow `lean-geometry.yml`, qui interroge les axiomes réellement utilisés. Le module du lac est né avec cette jambe câblée (tranche 2 de #18601).

Ce tableau est la réponse à la question posée en tête de carnet. Les deux méthodes ne s'évaluent pas sur une échelle unique « plus fort / plus faible » :

- l'algèbre **calcule** et produit un certificat qu'un tiers peut remultiplier, mais elle est relative à un encodage qu'elle ne peut pas auditer elle-même ;
- le noyau **établit** sans encodage et sans condition cachée, mais il ne calcule rien et ne dit rien des constructions.

Le pont n'est pas une hiérarchie : c'est une **répartition**. Le programme gradué de la série l'a suivie sans la nommer — 01 à 04 calculent, 05 établit, et l'escalier d'entrée fait passer de l'un à l'autre.

## Exercices

Les trois exercices prolongent la mesure. Ils sont à compléter ; le carnet doit s'exécuter de bout en bout même non complété.

In [8]:
# Exercice 1 — Le certificat de la conclusion symetrique.
# La conclusion MA^2 = MC^2 reduit elle aussi a zero (§2), mais son certificat
# n'est pas celui de §3. Calculez les cofacteurs de (MA2 - MC2) et verifiez par
# remultiplication, comme en §3. Les cofacteurs sont-ils les memes ?
# TODO etudiant
# Indice : base.reduce(sp.expand(MA2 - MC2)) rend (cofacteurs, reste) ; la
#   remultiplication s'ecrit sum(c * g.as_expr() for c, g in zip(cofacteurs, base.polys)).
# Etape 1 : obtenir cofacteurs_2, reste_2.
# Etape 2 : afficher les cofacteurs non nuls et l'ecart de remultiplication.
print("Exercice a completer")

Exercice a completer


In [9]:
# Exercice 2 — Lire le lac, compter ses enonces.
# Ecrivez une fonction qui, pour chaque fichier *.lean du dossier du lac, rend un
# tuple (nom du module, nombre d'enonces, nombre d'enonces prives). Le module
# MidpointHypotenuse en porte 4 dont 1 prive : votre fonction doit le confirmer.
# TODO etudiant
# Indice : os.walk sur le dossier "geometry_lean" ; reutilisez la regex ENONCE de §1 ;
#   sur Windows, triez avec sorted() pour un affichage stable.
# Etape 1 : parcourir les *.lean sous le dossier du lac.
# Etape 2 : compter, afficher une ligne par module.
print("Exercice a completer")

Exercice a completer

In [10]:
# Exercice 3 — Mesurer ce que porte le jeu d'hypotheses.
# §3 montrait que les cofacteurs font travailler les equations de milieu. Refaites
# le test de §2 en RETIRANT mid1 et mid2 de l'ideal : ne gardez que perp. Le reste
# de (MA2 - MB2) est-il encore nul ?
# TODO etudiant
# Indice : sp.groebner([perp], *VARS, order="lex", domain=sp.QQ) puis .reduce(...).
#   Regardez ensuite ce que contient le reste : y reconnait-on une quantite geometrique ?
# Etape 1 : construire la base reduite a perp.
# Etape 2 : afficher le reste et le comparer a l'expression de MA2 - MB2 hors ideal.
print("Exercice a completer")

Exercice a completer


## Conclusion

Ce carnet a fermé le chemin principal du programme gradué : les positions 01 à 04 calculent, la position 05 établit, et l'écart entre les deux régimes est ici mesuré plutôt que raconté.

| Brique | Rôle | Où |
|---|---|---|
| lecture du lac par fichier | les énoncés Lean sans build local | §1 |
| `sp.groebner(..., domain=sp.QQ)` | le verdict algébrique du fil rouge, trois restes nuls | §2 |
| certificat et remultiplication | ce que « reste nul » produit réellement | §3 |
| témoin négatif `MA² = MB²/4` | un polynôme résidu, non nul hors dégénérescence | §4 |
| rotation puis cisaillement | le verdict dépend de l'encodage, le noyau non | §5 |
| tableau des garanties | la répartition calcul / établissement | §6 |

Ce que la série laisse ouvert : la position **04b** du programme (`IMO-AG-30`, le proposeur neuronal couplé à DD+AR) reste en accrétion, hors du chemin principal — elle ne se lit pas sans ouvrir de lettre, comme l'exige la règle 2 du programme. Et le lac `geometry_lean` peut croître : ses énoncés sont le pendant formel de ce que les carnets Python calculent.

Voir aussi : [README de la série](README.md) · [Lean-38, l'escalier d'entrée](../../Lean-38-Capstone-Geometry-Lean-Python.ipynb) · [le lac `geometry_lean`](geometry_lean/README.md). Le programme gradué de la série est suivi sur l'EPIC #17544 ; le volet formel (re-parentage et lac) sur l'EPIC #18601.